# xLSTM — a toy-scale build of the matrix-memory LSTM (mLSTM)

A minimal implementation of the **mLSTM** cell from Beck et al., *"xLSTM:
Extended Long Short-Term Memory"* (2024) — the LSTM, redesigned with matrix
memory and exponential gating so it can be trained in parallel like a
transformer, while still running as a constant-memory recurrence at
inference time.

Companion write-up: `README.md` in this folder.

## 0. Setup

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

## 1. Why revisit the LSTM?

The classic LSTM updates a single *vector* memory cell using sigmoid input
and forget gates. It works, but it's fundamentally sequential — every step
depends on the previous one — which made it fall out of favor once
transformers showed that fully parallel training could scale much further.

xLSTM asks: what if we fix the LSTM's memory capacity and gating, keeping
the "recurrent state that decays and gets written to" idea, but redesign it
so that it *can* be reformulated in a parallel form for training (like the
other linear-attention-style layers in this repo)? The paper introduces two
variants; this notebook builds **mLSTM**, the fully parallelizable one:

- **Matrix memory** instead of a vector — the same idea GLA, RetNet, and KDA
  all use, a small matrix `C` that gets read out with a query.
- **Exponential gating** instead of plain sigmoid gating — the forget and
  input gates can now exceed what a sigmoid would allow, which the paper
  shows helps the LSTM "revise" earlier storage decisions.
- **A stabilizer** that tracks a running maximum in log-space, so the
  exponential gates don't blow up numerically over long sequences.

## 2. The update rule, step by step

At every timestep `t`:

1. Compute two **pre-activation gates** from the input: `i_tilde_t`
   (input gate) and `f_tilde_t` (forget gate) — both just linear
   projections, not yet squashed.
2. **Track a running log-scale maximum** `m_t`, combining the previous
   maximum (decayed by the forget gate, in log-space) with the current
   input gate:
   ```
   m_t = max(log_sigmoid(f_tilde_t) + m_{t-1}, i_tilde_t)
   ```
   This is the trick that keeps exponentiating the gates from ever
   overflowing — everything downstream is expressed *relative to* this
   running max.
3. Derive the **stabilized gates**:
   ```
   i_t = exp(i_tilde_t - m_t)
   f_t = exp(log_sigmoid(f_tilde_t) + m_{t-1} - m_t)
   ```
4. **Update the matrix memory** with an outer-product write (same shape of
   update as GLA/RetNet, but with these exponential gates):
   ```
   C_t = f_t * C_{t-1} + i_t * (k_t (x) v_t)
   n_t = f_t * n_{t-1} + i_t * k_t          # a normalizer vector
   ```
5. **Read out**, dividing by the normalizer to keep the output on a
   sensible scale:
   ```
   h_t = (C_t^T q_t) / max(|n_t · q_t|, 1)
   ```

> **Simplification used here:** this notebook implements mLSTM only — the
> paper's other variant, **sLSTM**, keeps a scalar (not matrix) memory with
> a "new memory mixing" mechanism, and is *not* fully parallelizable the way
> mLSTM is. mLSTM was chosen here because it slots into the same "linear
> attention with a matrix state" family as the rest of this repo, making it
> easy to compare side by side with GLA, RetNet, and KDA. The sequence is
> processed with a plain sequential loop for readability, same as
> everywhere else in this repo — a real xLSTM implementation would use a
> parallel formulation for training speed.

In [ ]:
class mLSTM(nn.Module):
    def __init__(self, d_model=64, n_heads=2, d_head=32):
        super().__init__()
        self.h, self.dh = n_heads, d_head
        inner = n_heads * d_head
        self.q_proj = nn.Linear(d_model, inner, bias=False)
        self.k_proj = nn.Linear(d_model, inner, bias=False)
        self.v_proj = nn.Linear(d_model, inner, bias=False)
        self.i_proj = nn.Linear(d_model, n_heads, bias=True)   # input gate pre-activation
        self.f_proj = nn.Linear(d_model, n_heads, bias=True)   # forget gate pre-activation
        self.o_proj = nn.Linear(d_model, inner, bias=True)     # output gate
        self.out_proj = nn.Linear(inner, d_model, bias=False)
        self.out_norm = nn.GroupNorm(n_heads, inner)

    def forward(self, x):
        B, T, D = x.shape
        H, Dh = self.h, self.dh
        q = self.q_proj(x).view(B, T, H, Dh) / (Dh ** 0.5)
        k = self.k_proj(x).view(B, T, H, Dh)
        v = self.v_proj(x).view(B, T, H, Dh)
        i_tilde = self.i_proj(x)                     # B,T,H
        log_f = F.logsigmoid(self.f_proj(x))          # B,T,H

        C = x.new_zeros(B, H, Dh, Dh)     # matrix memory
        n = x.new_zeros(B, H, Dh)          # normalizer
        m = x.new_full((B, H), -1e5)       # running log-scale stabilizer (step 2)

        outs = []
        for t in range(T):
            k_t, v_t, q_t = k[:, t], v[:, t], q[:, t]
            i_t_pre, logf_t = i_tilde[:, t], log_f[:, t]

            m_new = torch.maximum(logf_t + m, i_t_pre)             # step 2: running max
            i_act = torch.exp(i_t_pre - m_new)                      # step 3: stabilized input gate
            f_act = torch.exp(logf_t + m - m_new)                   # step 3: stabilized forget gate
            m = m_new

            C = f_act.view(B, H, 1, 1) * C + i_act.view(B, H, 1, 1) * k_t.unsqueeze(-1) * v_t.unsqueeze(-2)  # step 4
            n = f_act.view(B, H, 1) * n + i_act.view(B, H, 1) * k_t

            num = torch.einsum('bhd,bhde->bhe', q_t, C)             # step 5: read
            den = torch.clamp((q_t * n).sum(-1, keepdim=True).abs(), min=1.0)
            outs.append(num / den)

        o = torch.stack(outs, dim=1)
        o = self.out_norm(o.reshape(B * T, H * Dh)).reshape(B, T, H * Dh)
        gate = torch.sigmoid(self.o_proj(x))
        return self.out_proj(gate * o)

## 3. Assembling a tiny language model

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))
    def forward(self, x):
        norm = x.pow(2).mean(-1, keepdim=True)
        return x * torch.rsqrt(norm + self.eps) * self.weight

class SwiGLU(nn.Module):
    def __init__(self, d, hidden_mult=2):
        super().__init__()
        h = d * hidden_mult
        self.Wg = nn.Linear(d, h, bias=False)
        self.Wu = nn.Linear(d, h, bias=False)
        self.Wd = nn.Linear(h, d, bias=False)
    def forward(self, x):
        return self.Wd(F.silu(self.Wg(x)) * self.Wu(x))

class TinyLM(nn.Module):
    def __init__(self, vocab_size, d_model=64, n_layers=2):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.blocks = nn.ModuleList([mLSTM(d_model) for _ in range(n_layers)])
        self.mlps = nn.ModuleList([SwiGLU(d_model) for _ in range(n_layers)])
        self.norms1 = nn.ModuleList([RMSNorm(d_model) for _ in range(n_layers)])
        self.norms2 = nn.ModuleList([RMSNorm(d_model) for _ in range(n_layers)])
        self.final_norm = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, idx):
        x = self.embed(idx)
        for blk, mlp, n1, n2 in zip(self.blocks, self.mlps, self.norms1, self.norms2):
            x = x + blk(n1(x))
            x = x + mlp(n2(x))
        return self.lm_head(self.final_norm(x))

## Proving it actually works

Everything above is only worth something if gradients actually flow correctly
through mLSTM once it's wired into a real model. So the rest of this
notebook:

1. wraps mLSTM into a tiny 2-layer causal language model,
2. builds a **tiny synthetic dataset** (a repeating `"0123456789ABCDEF"`
   string — enough to check the model can learn *any* sequential structure
   at all, no real corpus needed),
3. runs **one forward + backward pass** as a sanity check (right output
   shape, no `NaN` gradients),
4. **trains for a few hundred steps**, and
5. **generates** from the trained model — if training worked, the output
   should show visible periodicity.

This is deliberately not a "real" training run. It exists purely to catch
architecture bugs, which is the whole point of a toy-scale build.

In [ ]:
# --- synthetic dataset ---
pattern = "0123456789ABCDEF"      # synthetic, no copyright concerns
text = pattern * 200
chars = sorted(set(text))
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for c, i in stoi.items()}
data = torch.tensor([stoi[c] for c in text], dtype=torch.long)
vocab_size = len(chars)
max_seq_len = 32

model = TinyLM(vocab_size).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"Model built. Trainable parameters: {n_params:,}")

In [ ]:
# --- sanity check: one forward + backward pass before training ---
xb0 = data[:max_seq_len].unsqueeze(0).to(device)
yb0 = data[1:max_seq_len + 1].unsqueeze(0).to(device)
out0 = model(xb0)
logits0 = out0[0] if isinstance(out0, tuple) else out0
print(f"Sanity check -- logits shape: {tuple(logits0.shape)} (expect [1, {max_seq_len}, {vocab_size}])")
loss0 = F.cross_entropy(logits0.reshape(-1, vocab_size), yb0.reshape(-1))
if isinstance(out0, tuple):
    loss0 = loss0 + out0[1]
loss0.backward()
n_nan_grads = sum(torch.isnan(p.grad).any().item() for p in model.parameters() if p.grad is not None)
print(f"Sanity check -- initial loss: {loss0.item():.4f}, NaN grads: {n_nan_grads}")
model.zero_grad()

In [ ]:
# --- training loop ---
def get_batch(data, block_size, batch_size, device):
    ix = torch.randint(0, len(data) - block_size - 1, (batch_size,))
    x = torch.stack([data[i:i + block_size] for i in ix])
    y = torch.stack([data[i + 1:i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)

opt = torch.optim.AdamW(model.parameters(), lr=3e-3)
n_steps, batch_size = 300, 16
print("Training on synthetic periodic sequence (verifies grads flow end-to-end)...")
for step in range(n_steps):
    xb, yb = get_batch(data, max_seq_len, batch_size, device)
    out = model(xb)
    logits = out[0] if isinstance(out, tuple) else out
    loss = F.cross_entropy(logits.reshape(-1, vocab_size), yb.reshape(-1))
    if isinstance(out, tuple):
        loss = loss + out[1]
    opt.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    opt.step()
    if step % 50 == 0 or step == n_steps - 1:
        print(f"  step {step:4d} | loss {loss.item():.4f}")

In [ ]:
# --- generation ---
@torch.no_grad()
def generate(model, start_idx, n_new):
    model.eval()
    idx = start_idx.clone()
    for _ in range(n_new):
        out = model(idx)
        logits = out[0] if isinstance(out, tuple) else out
        probs = F.softmax(logits[:, -1, :], dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)
        idx = torch.cat([idx, next_id], dim=1)
    model.train()
    return idx

start = data[:8].unsqueeze(0).to(device)
gen = generate(model, start, 48)[0].tolist()
print("Generated (should show visible periodicity if training worked):")
print(''.join(itos[i] for i in gen))

## Where to go from here

- **Implement sLSTM** alongside mLSTM and mix them in a block-wise stack,
  the way the xLSTM paper does — sLSTM's scalar memory with memory mixing
  gives it a different inductive bias than mLSTM's matrix memory.
- **Compare the stabilizer against KDA's `gmin` floor** — both exist to keep
  a gated recurrence numerically safe over long sequences, but they do it
  in different ways (a running log-max here vs. a hard floor on the decay
  there). Worth reading side by side.
- **Try a parallel (chunked) formulation** of the mLSTM recurrence for
  speed — same idea as the "where to go from here" sections in the other
  linear-attention notebooks in this repo.

Reference: Beck, Pöppel, Spanring, Auer, Prudnikova, Kopp, Klambauer,
Brandstetter, Hochreiter, *"xLSTM: Extended Long Short-Term Memory,"*
2024.